In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
bd.projects.set_current('propanol_production')

In [3]:
bd.databases

Databases dictionary with 10 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-image-SSP2-Base-2025
	ecoinvent-3.10-cutoff-image-SSP2-Base-2050
	ecoinvent-3.10-cutoff-image-SSP2-RCP19-2050
	ecoinvent-3.10-cutoff-image-SSP2-RCP26-2050
	inventories-ei310-Base-2025
	inventories-ei310-Base-2050
	inventories-ei310-RCP19-2050
	inventories-ei310-RCP26-2050

In [4]:
db = bd.Database('inventories-ei310-Base-2025')

In [5]:
method = ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')
method

('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')

In [6]:
def collect_recursive_calculation(activity, lcia_method, lca_obj = None, total_score = None, amount = 1, level = 0, max_level = 3, cutoff = 1e-2, collected_data = None, parent_activity = None):
    # initialize collected_data list if not provided
    if collected_data is None:
        collected_data = []

    # initialize LCA object if not provided
    if lca_obj is None:
        lca_obj = bc.LCA({activity: amount}, lcia_method)
        lca_obj.lci()
        lca_obj.lcia()
        total_score = lca_obj.score
    elif total_score is None:
        raise ValueError
    else:
        lca_obj.redo_lcia({activity.id: amount})
        if abs(lca_obj.score) <= abs(total_score * cutoff):
            return collected_data

    # collect data for the current activity
    if parent_activity is not None:  # skip adding entry for the root activity
        collected_data.append({
            'target': str(parent_activity),  # target is the parent activity
            'source': str(activity),  # source is the current activity
            'impact': lca_obj.score,  # relative contribution
            'level': level  # current level in the recursive calculation
        })

    # recursive case: drill down into the technosphere exchanges
    if level < max_level:
        for exc in activity.technosphere():
            collect_recursive_calculation(
                activity = exc.input,
                lcia_method = lcia_method,
                lca_obj = lca_obj,
                total_score = total_score,
                amount = amount * exc['amount'],
                level = level + 1,
                max_level = max_level,
                cutoff= cutoff,
                collected_data = collected_data,
                parent_activity = activity  # set current activity as parent for the next level
            )

    return collected_data, total_score

In [7]:
df_2020 = pd.DataFrame()

In [8]:
activity_names = [activity['name'] for activity in db if 'propanol production' in activity['reference product'] and 'GLO' in activity['location']]
activity_names

['propanol production, DAC carbon dioxide, wind hydrogen, green ethylene',
 'propanol production, fossil',
 'propanol production, DAC carbon dioxide, wind hydrogen, fossil ethylene',
 'propanol production, biogas, green ethylene',
 'propanol production, biogas, fossil ethylene']

In [9]:
activities = []
for ds in db:
    for activity_name in activity_names:
        if activity_name == ds['name'] and 'GLO' == ds['location']:
            activities.append(ds)
            break

for act in activities: 
    print(act['name'])
    collected_data, total_score = collect_recursive_calculation(
        activity = act,  # provide the appropriate activity object here
        lcia_method = method,  # Provide the appropriate LCIA method here
        max_level = 1,  # maximum level (level 1 is direct contributions)
        cutoff = 0,  # cut off is x% contribution
        amount = 1
    )

    df_2020 = pd.DataFrame(collected_data, columns=['target', 'source', 'impact', 'level'])
    df_2020.loc[len(df_2020)] = {'target': str(act), 'source': str(act), 'impact': total_score, 'level': 0}
    df_2020.sort_values(by = 'level', inplace=True)
    df_2020.to_excel(os.path.join('..', 'results', 'climate_change_breakdown', '2025', act['name'] + '_' + act['location'] + '.xlsx'))  

propanol production, DAC carbon dioxide, wind hydrogen, green ethylene
propanol production, DAC carbon dioxide, wind hydrogen, fossil ethylene
propanol production, biogas, green ethylene
propanol production, fossil
propanol production, biogas, fossil ethylene


In [10]:
db = bd.Database('inventories-ei310-RCP26-2050')

In [11]:
df_2050 = pd.DataFrame()

In [12]:
activity_names = [activity['name'] for activity in db if 'propanol production' in activity['reference product'] and 'GLO' in activity['location']]
activity_names

['propanol production, biogas, fossil ethylene',
 'propanol production, biogas, green ethylene',
 'propanol production, DAC carbon dioxide, wind hydrogen, green ethylene',
 'propanol production, fossil',
 'propanol production, DAC carbon dioxide, wind hydrogen, fossil ethylene']

In [13]:
activities = []
for ds in db:
    for activity_name in activity_names:
        if activity_name == ds['name'] and 'GLO' == ds['location']:
            activities.append(ds)
            break

for act in activities: 
    print(act['name'])
    collected_data, total_score = collect_recursive_calculation(
        activity = act,  # provide the appropriate activity object here
        lcia_method = method,  # Provide the appropriate LCIA method here
        max_level = 1,  # maximum level (level 1 is direct contributions)
        cutoff = 0,  # cut off is x% contribution
        amount = 1
    )

    df_2050 = pd.DataFrame(collected_data, columns=['target', 'source', 'impact', 'level'])
    df_2050.loc[len(df_2050)] = {'target': str(act), 'source': str(act), 'impact': total_score, 'level': 0}
    df_2050.sort_values(by = 'level', inplace=True)
    df_2050.to_excel(os.path.join('..', 'results', 'climate_change_breakdown', '2050', act['name'] + '_' + act['location'] + '.xlsx'))  

propanol production, fossil


propanol production, biogas, green ethylene
propanol production, DAC carbon dioxide, wind hydrogen, fossil ethylene
propanol production, biogas, fossil ethylene
propanol production, DAC carbon dioxide, wind hydrogen, green ethylene
